In [ ]:
import arcpy
import pandas as pd
import numpy as np
import re

In [ ]:
# setting workspace
arcpy.env.workspace = "YOUR/PATH/TO/industrial-pollution-risk.gdb"
arcpy.env.overwriteOutput = True

## Most at-risk schools
The schools with the highest underlying toxicity concentration values.

In [ ]:
# converting to dataframe
SCH_rsei = pd.DataFrame(arcpy.da.FeatureClassToNumPyArray("SCH_rsei", [
    "LEAID",
    "NAME",
    "CITY",
    "STATE",
    "ZIP",
    "LAT",
    "LON",
    "ToxConc",
    "ToxConcP"]
))

# cleaning & sorting by underlying toxconc
SCH_rsei = SCH_rsei.rename(columns={"NAME": "Name"})
SCH_rsei.Name = SCH_rsei.Name.str.title()
SCH_rsei["Location"] = SCH_rsei.CITY.str.title() + ", " + SCH_rsei.STATE
SCH_rsei = SCH_rsei.sort_values("ToxConc", ascending=False)

# exporting top 10 schools
SCH_rsei[["Name", "Location", "ToxConc", "ToxConcP"]][:10].to_csv("top10sch.csv")

# printing top 30 schools
SCH_rsei[:30]

In [ ]:
# top 10 cities where there are the most schools in the 95th percentile of pollution
SCH_rsei[SCH_rsei.ToxConcP>=0.95].groupby(["Location"]).count()["Name"].sort_values(ascending=False)[:10]

In [ ]:
# top 10 states where there are the most schools in the 95th percentile of pollution
SCH_rsei[SCH_rsei.ToxConcP>=0.95].groupby("STATE").count()["Name"].sort_values(ascending=False)[:10]

## Worst polluters within 5 km of a school

### Worst industrial facilities
The top facilities operating within 5 km of a school (ranked by emissions).

In [ ]:
# converting layer of sites within 5km of a school to a dataframe
TRI_5km = pd.DataFrame(arcpy.da.FeatureClassToNumPyArray("TRI_5km", [
    "TRIFD",
    "FACILITY_NAME", 
    "CITY",
    "ST",
    "ON_SITE_RELEASE_TOTAL", 
    "STANDARD_PARENT_CO_NAME",
    "FEDERAL_FACILITY",
    "INDUSTRY_SECTOR"]
))

# sorting by total on-site releases
TRI_5km = TRI_5km.sort_values('ON_SITE_RELEASE_TOTAL', ascending=False)
TRI_5km['Location'] = TRI_5km.CITY.str.title() + ', ' + TRI_5km.ST

# cleaning list of top 10 sites
top10sites = TRI_5km[['FACILITY_NAME', 'Location','ON_SITE_RELEASE_TOTAL', 'INDUSTRY_SECTOR']][:10]
top10sites = top10sites.rename(columns={"FACILITY_NAME":"Facility", "ON_SITE_RELEASE_TOTAL":"Emissions", "INDUSTRY_SECTOR":"Sector"})
top10sites.Facility = top10sites.Facility.str.title()
top10sites['Emissions'] = (np.round(top10sites['Emissions']/2000/1e3)).astype(int) # to tons
top10sites['Emissions'] = [str(x) + 'k tons' for x in top10sites['Emissions']]

# exporting and printing top 10 sites
top10sites.to_csv('top10sites.csv')
top10sites

### Worst parent companies
The top parent companies of facilities operating within 5 km of a school (ranked by emissions).

In [ ]:
# converting total emissions across all sites to a dataframe
TRI_XY = pd.DataFrame(arcpy.da.FeatureClassToNumPyArray("TRI_XY", [
    "TRIFD",
    "FACILITY_NAME", 
    "ON_SITE_RELEASE_TOTAL", 
    "STANDARD_PARENT_CO_NAME",
    "FEDERAL_FACILITY",
    "INDUSTRY_SECTOR"]
))

# grouping by parent company and sorting by total on-site releases
comps = TRI_XY.groupby('STANDARD_PARENT_CO_NAME')['ON_SITE_RELEASE_TOTAL'].agg(
    Emissions='sum', Facilities='count').sort_values('Emissions', ascending=False)

In [ ]:
# grouping <5 km site data set by parent company and sorting by total on-site releases
comps_5km = TRI_5km.groupby('STANDARD_PARENT_CO_NAME')['ON_SITE_RELEASE_TOTAL'].agg(
    Emissions='sum', Facilities='count').sort_values('Emissions', ascending=False)

# joining top polluter data with full country data to calculate share of emissions and facilities in the <5 km zone
comps_5km = comps_5km.join(comps, rsuffix=' total')
comps_5km['% of Total Emissions'] = comps_5km['Emissions'] / comps_5km['Emissions total'] * 100
comps_5km['% of Total Facilities'] = comps_5km['Facilities'] / comps_5km['Facilities total'] * 100
comps_5km = comps_5km[['Emissions', 'Facilities', '% of Total Emissions', '% of Total Facilities']]

# cleaning list of top 10 companies
top10comps = comps_5km[:10]
top10comps.index = [re.sub(' Inc$| Co$| Corp$| Llc$| Holdings$', '', x.title()) for x in top10comps.index]
top10comps.loc[:,'Emissions'] = (np.round(top10comps['Emissions']/2000/1e3)).astype(int) # to tons
top10comps.loc[:,'Emissions'] = [str(x) + 'k tons' for x in top10comps['Emissions']]
top10comps.loc[:,'% of Total Emissions'] = np.round(top10comps['% of Total Emissions']).astype(int)
top10comps.loc[:,'% of Total Emissions'] = [str(x) + '%' for x in top10comps['% of Total Emissions']]
top10comps.loc[:,'% of Total Facilities'] = np.round(top10comps['% of Total Facilities']).astype(int)
top10comps.loc[:,'% of Total Facilities'] = [str(x) + '%' for x in top10comps['% of Total Facilities']]

# exporting and printing top 10 companies
top10comps.to_csv('top10comps.csv')
top10comps